# Baseline Modeling (`Lexicon`, `BERT-Linear`, `BERT-CRF`)

Notebook ini memvalidasi pipeline modeling end-to-end pada `bert-base-uncased` (`BIO`, `MAX_LEN = 256`) di `tsd_train` dan `tsd_trial` (`tsd_test` tidak disentuh):
1. **Baseline 1:** Rule-Based Lexicon Lookup dari `tsd_train`.
2. **Quality Gate Wajib:** *Single-Batch Overfit Sanity Test* (8 kalimat, 30 iterasi, target F1 > 0,95).
3. **Baseline 2 vs Model Utama:** `BERT + Linear Head` vs `BERT + CRF` (*differential learning rate*, *early stopping* berbasis `tsd_trial` Char F1).
4. **Quality Gate Transisi & Ringkasan Metrik:** Verifikasi matriks transisi CRF dan penyimpanan prediksi `tsd_trial` ke `outputs/task_3/` untuk *post-processing*.

In [1]:
from collections import Counter
import gc
import json
from pathlib import Path
import re
import sys
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchcrf import CRF
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src" / "task_3").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.task_3.config import (
    DEFAULT_BACKBONE,
    ID2LABEL,
    MAX_LEN,
    NUM_TAGS,
    RESULTS_DIR,
    RUNS_DIR,
    SEEDS,
    set_seed,
)
from src.task_3.dataset import ToxicSpanCollator, ToxicSpanDataset, bio_to_char_spans
from src.task_3.metrics import evaluate_bio_tags, evaluate_char_spans


def log(tag, message, *, err=None):
    if err is None:
        err = tag in ("ERROR", "FATAL", "WARN")
    print(f"[{tag}] {message}", file=sys.stderr if err else sys.stdout, flush=True)


def print_table(title, rows, line_width=49):
    print(title)
    widths = [max(len(str(c)) for c in col) for col in zip(*rows)]
    total = sum(widths) + 3 * (len(widths) - 1)
    print("-" * max(line_width, total))
    for row in rows:
        print(" | ".join(str(c).ljust(w) for c, w in zip(row, widths)).rstrip())


set_seed(SEEDS[0])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_BASE = RESULTS_DIR / "baseline"
CKPT_DIR = RUNS_DIR / "baseline"
OUT_BASE.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_BACKBONE, use_fast=True)
train_ds = ToxicSpanDataset("train", tokenizer, max_length=MAX_LEN)
trial_ds = ToxicSpanDataset("trial", tokenizer, max_length=MAX_LEN)
collator = ToxicSpanCollator(pad_token_id=tokenizer.pad_token_id)

log("START", f"Loaded train={len(train_ds)}, trial={len(trial_ds)}, backbone={DEFAULT_BACKBONE}, device={DEVICE}")

[START] Loaded train=7939, trial=690, backbone=bert-base-uncased, device=cuda


## 1. Baseline 1 — Lexicon Lookup

In [2]:
WORD_RE = re.compile(r"\b\w+\b")


def build_toxic_lexicon(rows, min_freq=3, min_ratio=0.5):
    total_cnt = Counter()
    toxic_cnt = Counter()
    for row in rows:
        span_set = set(row["spans"])
        for m in WORD_RE.finditer(row["text"]):
            word = m.group(0).lower()
            total_cnt[word] += 1
            tok_len = m.end() - m.start()
            overlap = sum(pos in span_set for pos in range(m.start(), m.end()))
            if overlap * 2 >= tok_len:
                toxic_cnt[word] += 1
    return {
        word
        for word, tc in toxic_cnt.items()
        if tc >= min_freq and (tc / total_cnt[word]) >= min_ratio
    }


def predict_lexicon(text, lexicon):
    spans = []
    for m in WORD_RE.finditer(text):
        if m.group(0).lower() in lexicon:
            spans.extend(range(m.start(), m.end()))
    return spans


lexicon = build_toxic_lexicon(train_ds.rows, min_freq=3, min_ratio=0.5)
t0 = time.perf_counter()
lex_preds = [predict_lexicon(r["text"], lexicon) for r in trial_ds.rows]
lex_latency_ms = (time.perf_counter() - t0) * 1000.0 / len(trial_ds)

lex_stats = evaluate_char_spans(lex_preds, [r["spans"] for r in trial_ds.rows])
(OUT_BASE / "lexicon_trial_preds.json").write_text(json.dumps(lex_preds), encoding="utf-8")

log("CHECKPOINT 1", f"Lexicon size={len(lexicon)} kata | Trial Char F1={lex_stats['f1']:.5f}")

[CHECKPOINT 1] Lexicon size=221 kata | Trial Char F1=0.59794


## 2. Definisi Model (`BertLinear` & `BertCRF`) dan Helper Training

In [3]:
class BertLinear(nn.Module):
    def __init__(self, model_name, num_tags=NUM_TAGS, dropout=0.2):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, num_tags)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask.long())
        emissions = self.classifier(self.dropout(outputs.last_hidden_state))
        if labels is not None:
            masked_labels = labels.masked_fill(~attention_mask, -100)
            return self.loss_fn(emissions.view(-1, emissions.size(-1)), masked_labels.view(-1))
        pred_ids = emissions.argmax(dim=-1)
        lengths = attention_mask.sum(dim=1).tolist()
        return [seq[:length].tolist() for seq, length in zip(pred_ids, lengths)]


class BertCRF(nn.Module):
    def __init__(self, model_name, num_tags=NUM_TAGS, dropout=0.2):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, num_tags)
        self.crf = CRF(num_tags=num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask.long())
        emissions = self.classifier(self.dropout(outputs.last_hidden_state))
        if labels is not None:
            return -self.crf(emissions, labels, mask=attention_mask, reduction="mean")
        return self.crf.decode(emissions, mask=attention_mask)


def build_optimizer(model, encoder_lr=3e-5, head_lr=1e-3):
    groups = [
        {"params": model.encoder.parameters(), "lr": encoder_lr, "weight_decay": 0.01},
        {"params": model.classifier.parameters(), "lr": head_lr, "weight_decay": 0.01},
    ]
    if hasattr(model, "crf"):
        groups.append({"params": model.crf.parameters(), "lr": head_lr, "weight_decay": 0.0})
    return torch.optim.AdamW(groups)


def evaluate_model(model, loader, ds):
    model.eval()
    pred_seqs = []
    with torch.no_grad():
        for batch in loader:
            pred_seqs.extend(
                model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
            )

    pred_spans = [
        bio_to_char_spans(p_seq, sample["offset_mapping"], row["text"])
        for p_seq, sample, row in zip(pred_seqs, ds.samples, ds.rows)
    ]
    char_stats = evaluate_char_spans(pred_spans, [r["spans"] for r in ds.rows])
    pred_tags = [[ID2LABEL[t] for t in p_seq] for p_seq in pred_seqs]
    gold_tags = [[ID2LABEL[t] for t in sample["labels"]] for sample in ds.samples]
    seq_stats = evaluate_bio_tags(pred_tags, gold_tags)
    return char_stats, seq_stats, pred_spans, pred_tags


def measure_latency_ms(model, ds, n_warmup=20):
    model.eval()
    loader = DataLoader(ds, batch_size=1, shuffle=False, collate_fn=collator)
    items = list(loader)
    with torch.no_grad():
        for batch in items[:n_warmup]:
            _ = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for batch in items:
            _ = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000.0 / len(items)

## 3. Quality Gate: Single-Batch Overfit Sanity Test (`BertCRF`)

In [4]:
set_seed(SEEDS[0])
overfit_ds = Subset(train_ds, list(range(8)))
overfit_loader = DataLoader(overfit_ds, batch_size=8, shuffle=False, collate_fn=collator)
overfit_batch = next(iter(overfit_loader))
overfit_golds = [train_ds.rows[i]["spans"] for i in range(8)]

sanity_model = BertCRF(DEFAULT_BACKBONE).to(DEVICE)
sanity_opt = build_optimizer(sanity_model)

input_ids = overfit_batch["input_ids"].to(DEVICE)
mask = overfit_batch["attention_mask"].to(DEVICE)
labels = overfit_batch["labels"].to(DEVICE)

losses = []
sanity_model.train()
for step in range(1, 31):
    sanity_opt.zero_grad()
    loss = sanity_model(input_ids, mask, labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(sanity_model.parameters(), 1.0)
    sanity_opt.step()
    losses.append(loss.item())

sanity_model.eval()
with torch.no_grad():
    overfit_seqs = sanity_model(input_ids, mask)
overfit_spans = [
    bio_to_char_spans(p_seq, train_ds.samples[i]["offset_mapping"], train_ds.rows[i]["text"])
    for i, p_seq in enumerate(overfit_seqs)
]
overfit_f1 = evaluate_char_spans(overfit_spans, overfit_golds)["f1"]

print_table(
    "Single-Batch Overfit Check (8 kalimat, 30 iterasi)",
    [
        ["Metrik", "Nilai"],
        ["Initial loss (step 1)", f"{losses[0]:.4f}"],
        ["Final loss (step 30)", f"{losses[-1]:.4f}"],
        ["Overfit Char F1", f"{overfit_f1:.6f}"],
    ],
)
assert overfit_f1 > 0.95, f"Single-batch overfit failed: F1={overfit_f1:.4f} <= 0.95"
log("CHECKPOINT 2", "Single-batch overfit sanity test LULUS (F1 > 0.95)")

del sanity_model, sanity_opt, input_ids, mask, labels
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

W1121 14:58:42.678000 29820 venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\ASUS\Desktop\nlp\tubes\IF5153-Chat-Moderation\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enab

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Users\ASUS\Desktop\nlp\tubes\IF5153-Chat-Moderation\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Single-Batch Overfit Check (8 kalimat, 30 iterasi)
-------------------------------------------------
Metrik                | Nilai
Initial loss (step 1) | 74.8908
Final loss (step 30)  | 0.0196
Overfit Char F1       | 1.000000
[CHECKPOINT 2] Single-batch overfit sanity test LULUS (F1 > 0.95)


## 4. Training & Evaluasi `BERT-Linear` vs `BERT-CRF` (`tsd_trial`)

In [5]:
def train_and_eval(run_name, model_cls, epochs=5, batch_size=16):
    set_seed(SEEDS[0])
    tr_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collator)
    va_loader = DataLoader(trial_ds, batch_size=batch_size, shuffle=False, collate_fn=collator)

    model = model_cls(DEFAULT_BACKBONE).to(DEVICE)
    optimizer = build_optimizer(model)
    total_steps = len(tr_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(0.1 * total_steps)),
        num_training_steps=total_steps,
    )

    ckpt_path = CKPT_DIR / f"{run_name}.pt"
    best_f1 = -1.0
    best_epoch = 0
    history = []

    for epoch in range(1, epochs + 1):
        t_ep = time.perf_counter()
        model.train()
        total_loss = 0.0
        for batch in tr_loader:
            optimizer.zero_grad()
            loss = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE),
                batch["labels"].to(DEVICE),
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(tr_loader)
        char_stats, seq_stats, _, _ = evaluate_model(model, va_loader, trial_ds)
        dt = time.perf_counter() - t_ep
        history.append({
            "epoch": epoch,
            "train_loss": avg_loss,
            "trial_char_f1": char_stats["f1"],
            "trial_empty_f1": char_stats["empty_f1"],
            "trial_non_empty_f1": char_stats["non_empty_f1"],
            "trial_seq_f1": seq_stats["f1"],
            "sec": dt,
        })
        log(
            "CHECKPOINT " + str(epoch),
            f"{run_name} ep {epoch}/{epochs} | loss={avg_loss:.4f} | trial_f1={char_stats['f1']:.5f} | empty={char_stats['empty_f1']:.4f} | seq_f1={seq_stats['f1']:.5f} ({dt:.1f}s)",
        )
        if char_stats["f1"] > best_f1:
            best_f1 = char_stats["f1"]
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    char_stats, seq_stats, pred_spans, pred_tags = evaluate_model(model, va_loader, trial_ds)
    latency_ms = measure_latency_ms(model, trial_ds)
    transitions = model.crf.transitions.detach().cpu().clone() if hasattr(model, "crf") else None

    (OUT_BASE / f"{run_name}_trial_preds.json").write_text(
        json.dumps({"spans": pred_spans, "bio_tags": pred_tags}),
        encoding="utf-8",
    )

    del model, optimizer, scheduler
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "best_epoch": best_epoch,
        "char_stats": char_stats,
        "seq_stats": seq_stats,
        "latency_ms": latency_ms,
        "history": history,
        "transitions": transitions,
    }


linear_res = train_and_eval("bert_linear", BertLinear, epochs=5, batch_size=16)
crf_res = train_and_eval("bert_crf", BertCRF, epochs=5, batch_size=16)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[CHECKPOINT 1] bert_linear ep 1/5 | loss=0.3036 | trial_f1=0.62639 | empty=0.1860 | seq_f1=0.57783 (236.3s)
[CHECKPOINT 2] bert_linear ep 2/5 | loss=0.2173 | trial_f1=0.63499 | empty=0.1163 | seq_f1=0.57738 (255.3s)
[CHECKPOINT 3] bert_linear ep 3/5 | loss=0.1752 | trial_f1=0.66695 | empty=0.0930 | seq_f1=0.53992 (258.6s)
[CHECKPOINT 4] bert_linear ep 4/5 | loss=0.1159 | trial_f1=0.67110 | empty=0.0465 | seq_f1=0.54688 (262.9s)
[CHECKPOINT 5] bert_linear ep 5/5 | loss=0.0810 | trial_f1=0.67048 | empty=0.0233 | seq_f1=0.54545 (260.0s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[CHECKPOINT 1] bert_crf ep 1/5 | loss=14.2261 | trial_f1=0.66121 | empty=0.1163 | seq_f1=0.59067 (444.7s)
[CHECKPOINT 2] bert_crf ep 2/5 | loss=7.9149 | trial_f1=0.68011 | empty=0.0698 | seq_f1=0.60203 (404.6s)
[CHECKPOINT 3] bert_crf ep 3/5 | loss=5.8321 | trial_f1=0.67433 | empty=0.1163 | seq_f1=0.60789 (431.4s)
[CHECKPOINT 4] bert_crf ep 4/5 | loss=4.0340 | trial_f1=0.68038 | empty=0.0233 | seq_f1=0.58598 (445.9s)
[CHECKPOINT 5] bert_crf ep 5/5 | loss=2.8022 | trial_f1=0.67595 | empty=0.0233 | seq_f1=0.59106 (566.8s)


## 5. Verifikasi Matriks Transisi CRF & Perbandingan Akhir Baseline (`tsd_trial`)

In [6]:
labels = [ID2LABEL[i] for i in range(NUM_TAGS)]
mat = crf_res["transitions"].numpy()
trans_rows = [["From \\ To"] + labels]
for i, src_lbl in enumerate(labels):
    trans_rows.append([src_lbl] + [f"{mat[i, j]:+.3f}" for j in range(NUM_TAGS)])

print_table("Matriks Transisi CRF (BertCRF Terbaik)", trans_rows)
assert mat[0, 2] < mat[1, 2] and mat[0, 2] < mat[0, 1], "CRF transition check failed: O->I must be lower than B->I and O->B"
log("CHECKPOINT 3", "Verifikasi matriks transisi CRF LULUS (O->I < B->I dan O->I < O->B)")

summary_rows = [
    ["Model", "BestEp", "Char F1", "SEM", "Empty F1", "NonEmpty F1", "Seqeval F1", "Latency(ms)"],
    [
        "Lexicon Baseline",
        "-",
        f"{lex_stats['f1']:.5f}",
        f"{lex_stats['sem']:.5f}",
        f"{lex_stats['empty_f1']:.5f}",
        f"{lex_stats['non_empty_f1']:.5f}",
        "-",
        f"{lex_latency_ms:.3f}",
    ],
    [
        "BERT-Linear",
        str(linear_res["best_epoch"]),
        f"{linear_res['char_stats']['f1']:.5f}",
        f"{linear_res['char_stats']['sem']:.5f}",
        f"{linear_res['char_stats']['empty_f1']:.5f}",
        f"{linear_res['char_stats']['non_empty_f1']:.5f}",
        f"{linear_res['seq_stats']['f1']:.5f}",
        f"{linear_res['latency_ms']:.2f}",
    ],
    [
        "BERT-CRF",
        str(crf_res["best_epoch"]),
        f"{crf_res['char_stats']['f1']:.5f}",
        f"{crf_res['char_stats']['sem']:.5f}",
        f"{crf_res['char_stats']['empty_f1']:.5f}",
        f"{crf_res['char_stats']['non_empty_f1']:.5f}",
        f"{crf_res['seq_stats']['f1']:.5f}",
        f"{crf_res['latency_ms']:.2f}",
    ],
]
print()
print_table("Perbandingan Baseline Phase 4 (tsd_trial)", summary_rows)

summary_payload = {
    "lexicon": {"char": lex_stats, "latency_ms": lex_latency_ms},
    "bert_linear": {
        "best_epoch": linear_res["best_epoch"],
        "char": linear_res["char_stats"],
        "seqeval": linear_res["seq_stats"],
        "latency_ms": linear_res["latency_ms"],
        "history": linear_res["history"],
    },
    "bert_crf": {
        "best_epoch": crf_res["best_epoch"],
        "char": crf_res["char_stats"],
        "seqeval": crf_res["seq_stats"],
        "latency_ms": crf_res["latency_ms"],
        "history": crf_res["history"],
    },
}
(OUT_BASE / "baseline_summary.json").write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
log("DONE", f"Hasil baseline tersimpan di {OUT_BASE}")

Matriks Transisi CRF (BertCRF Terbaik)
-------------------------------------------------
From \ To | O      | B-TOXIC | I-TOXIC
O         | +0.492 | +0.411  | -0.768
B-TOXIC   | -0.557 | -0.329  | +0.718
I-TOXIC   | -0.725 | -0.636  | +0.821
[CHECKPOINT 3] Verifikasi matriks transisi CRF LULUS (O->I < B->I dan O->I < O->B)

Perbandingan Baseline Phase 4 (tsd_trial)
-------------------------------------------------------------------------------------------------
Model            | BestEp | Char F1 | SEM     | Empty F1 | NonEmpty F1 | Seqeval F1 | Latency(ms)
Lexicon Baseline | -      | 0.59794 | 0.01545 | 0.09302  | 0.63150     | -          | 0.028
BERT-Linear      | 4      | 0.67110 | 0.01369 | 0.04651  | 0.71261     | 0.54688    | 28.38
BERT-CRF         | 4      | 0.68038 | 0.01370 | 0.02326  | 0.72405     | 0.58598    | 36.86
[DONE] Hasil baseline tersimpan di C:\Users\ASUS\Desktop\nlp\tubes\IF5153-Chat-Moderation\outputs\task_3\results\baseline
